# ACE-Net — Stage-1 Emotion Eval (Tables 2/3)  ·  PERSON B

Evaluates the trained Stage-1 emotion extractors on their held-out test splits
and prints **Accuracy + Weighted-F1 + confusion matrix** per dataset
(paper Tables 2 and 3). No training — fast.

Set Runtime → T4 GPU (or even CPU works, just slower).

**You need uploaded:** `emotion_vectors.zip` on Drive (CREMA genuine + MELD),
and all four Stage-1 checkpoints (visual/speech x crema/meld).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Clone repo

In [ ]:
%cd /content
!rm -rf Baseline_Training
!git clone https://github.com/gjvlio/Baseline_Training.git
%cd Baseline_Training
!git log --oneline -1

## Install dependencies

In [ ]:
!pip -q install torch torchvision torchaudio transformers librosa pillow
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## Get the vectors

Mount Drive and unzip **`emotion_vectors.zip`** (upload it to your Drive root first).
Zip contains `CREMA-D/GENUINE_LastHalf`, `CREMA-D/GENUINE_FirstHalf`, and `MELD/` (with train/dev/test) at the right nesting so they unzip to `data/CREMA-D/...` and `data/MELD/...`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
ZIP = '/content/drive/MyDrive/emotion_vectors.zip'   # adjust if needed
DST = '/content/Baseline_Training/data'
os.makedirs(DST, exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(DST)
print('unzipped under data/:')
for root,dirs,_ in os.walk(DST):
    depth = root[len(DST):].count(os.sep)
    if depth < 2: print(' ', root)

## Upload Stage-1 checkpoints

In [ ]:
import os
os.makedirs('checkpoints', exist_ok=True)
from google.colab import files
up = files.upload()
for name in up:
    os.replace(name, f'checkpoints/{name}')
print(os.listdir('checkpoints'))

## Table 3 — Facial emotion (FV-LiteNet)

In [ ]:
!cd /content/Baseline_Training && PYTHONPATH=. python -m src.eval_stage1 --branch visual --dataset crema
!cd /content/Baseline_Training && PYTHONPATH=. python -m src.eval_stage1 --branch visual --dataset meld

## Table 2 — Speech-Text emotion (MDCNN + cross-attention)

In [ ]:
!cd /content/Baseline_Training && PYTHONPATH=. python -m src.eval_stage1 --branch speech_text --dataset crema
!cd /content/Baseline_Training && PYTHONPATH=. python -m src.eval_stage1 --branch speech_text --dataset meld

## Notes

These numbers come from time-capped Stage-1 training (few epochs on a 6GB
local GPU), so accuracy is under-converged vs the paper. They are valid,
just not fully trained. For better numbers, retrain Stage-1 longer on the T4
(`python -m src.train_stage1 --branch ... --dataset ... --epochs 40`).

## Results — tables + confusion-matrix figures

Runs each Stage-1 model inline (held-out test split), builds a summary table
(paper Tables 2/3: ACC + Weighted-F1) and renders **confusion-matrix heatmaps**
(paper Figures 5/6). Figures + a CSV are saved to Drive
`MyDrive/acenet_results/` so you can drop them straight into the report.

In [ ]:
import sys, os
sys.path.insert(0, '/content/Baseline_Training')
os.chdir('/content/Baseline_Training')
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='white')
from torch.utils.data import DataLoader
from src import config
from src.config import TrainConfig
from src.data import manifests
from src.data.dataset import EmotionDataset, collate_emotion
from src.models.speech_text import SpeechTextModule
from src.models.fv_litenet import FVLiteNet
from src.train_utils import set_seed, stratified_split
from src.eval_stage1 import _collect_preds, metrics as emo_metrics, confusion as emo_conf

cfg = TrainConfig(); set_seed(cfg.seed)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = '/content/drive/MyDrive/acenet_results'; os.makedirs(OUT, exist_ok=True)

def run_eval(branch, dataset):
    tag = f'{branch}_{dataset}'
    ckpt = config.CKPT_ROOT / f'stage1_{tag}.pt'
    if not ckpt.exists():
        return None
    names = config.EMOTION_DATASETS[dataset]['emotions']; nc = len(names)
    samples = manifests.build_emotion_samples(dataset)
    _,_,test_s = stratified_split(samples, key_fn=lambda s: s.emotion,
        ratios=(cfg.train_ratio,cfg.val_ratio,cfg.test_ratio), seed=cfg.seed)
    model = (SpeechTextModule(n_classes=nc) if branch=='speech_text'
             else FVLiteNet(n_classes=nc)).to(DEV)
    model.load_state_dict(torch.load(ckpt, map_location=DEV))
    dl = DataLoader(EmotionDataset(test_s), batch_size=16, shuffle=False,
                    num_workers=2, collate_fn=collate_emotion)
    y,p = _collect_preds(model, dl, branch, DEV)
    acc,wf1 = emo_metrics(y,p,nc)
    return dict(branch=branch, dataset=dataset, names=names,
                acc=acc*100, wf1=wf1, y=y, p=p, n=len(y))

RESULTS = {}
for branch in ['visual','speech_text']:
    for ds in ['crema','meld']:
        r = run_eval(branch, ds)
        if r: RESULTS[(branch,ds)] = r
print('evaluated:', list(RESULTS.keys()))

### Summary table (Tables 2 & 3)

In [ ]:
rows = []
label = {'visual':'FV-LiteNet (Table 3)','speech_text':'MDCNN (Table 2)'}
for (br,ds),r in RESULTS.items():
    rows.append({'Model':label[br],'Dataset':ds.upper(),
                 'ACC %':round(r['acc'],2),'Weighted-F1':round(r['wf1'],3),'n':r['n']})
df = pd.DataFrame(rows).sort_values(['Model','Dataset']).reset_index(drop=True)
df.to_csv(f'{OUT}/stage1_summary.csv', index=False)
print('saved', f'{OUT}/stage1_summary.csv')
df

### Bar chart — ACC vs Weighted-F1

In [ ]:
if not df.empty:
    fig,ax = plt.subplots(figsize=(8,4.5))
    x = np.arange(len(df)); w=0.38
    ax.bar(x-w/2, df['ACC %'], w, label='ACC %', color='#4C72B0')
    ax.bar(x+w/2, df['Weighted-F1']*100, w, label='Weighted-F1 x100', color='#DD8452')
    ax.set_xticks(x); ax.set_xticklabels([f"{r.Dataset}\n{r.Model.split()[0]}" for r in df.itertuples()], fontsize=8)
    ax.set_ylabel('score'); ax.set_ylim(0,100); ax.legend(); ax.set_title('Stage-1 Emotion Recognition')
    for i,(a,f) in enumerate(zip(df['ACC %'],df['Weighted-F1']*100)):
        ax.text(i-w/2,a+1,f'{a:.1f}',ha='center',fontsize=7); ax.text(i+w/2,f+1,f'{f:.1f}',ha='center',fontsize=7)
    plt.tight_layout(); plt.savefig(f'{OUT}/stage1_bars.png', dpi=150, bbox_inches='tight'); plt.show()
    print('saved', f'{OUT}/stage1_bars.png')

### Confusion-matrix heatmaps (Figures 5 & 6)

Row-normalized. Saved individually as PNGs to Drive.

In [ ]:
for (br,ds),r in RESULTS.items():
    names=r['names']; cm=emo_conf(r['y'],r['p'],len(names)).astype(float)
    cmn = cm / cm.sum(1,keepdims=True).clip(min=1)
    fig,ax=plt.subplots(figsize=(1+0.8*len(names), 0.8*len(names)))
    sns.heatmap(cmn, annot=True, fmt='.2f', cmap='Blues', cbar=True,
                xticklabels=names, yticklabels=names, ax=ax, vmin=0, vmax=1)
    title=f"{'FV-LiteNet' if br=='visual' else 'MDCNN'} — {ds.upper()}  (ACC {r['acc']:.1f}%)"
    ax.set_title(title); ax.set_xlabel('predicted'); ax.set_ylabel('true')
    fn=f'{OUT}/confusion_{br}_{ds}.png'
    plt.tight_layout(); plt.savefig(fn, dpi=150, bbox_inches='tight'); plt.show()
    print('saved', fn)

### All results saved to Drive

`MyDrive/acenet_results/` now contains:
- `stage1_summary.csv` — the Table 2/3 numbers
- `stage1_bars.png` — ACC / F1 bar chart
- `confusion_<branch>_<dataset>.png` — one heatmap per model

Download from Drive for the report.